# Hotel Reviews Sentiment Analysis
## Sentiment Analysis using ML and DL

This notebook contains:
- Logistic Regression (Machine Learning)
- LSTM Neural Network (Deep Learning)
- Text preprocessing
- TF-IDF Vectorization
- Model evaluation

In [ ]:
# Install required libraries (Run if needed)
# !pip install pandas numpy scikit-learn tensorflow nltk

In [ ]:
# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(
    "hotel_reviews.csv",
    engine='python',
    on_bad_lines='skip'
)

print(df.head())
print(df.columns)

## Note
Change the column names below according to your dataset.

- Review column example: `Review`
- Sentiment column example: `Sentiment`

In [ ]:
# =========================
# SELECT COLUMNS
# =========================

text_column = "Review"
target_column = "Sentiment"

# Keep only required columns
df = df[[text_column, target_column]]

# Remove missing values
df.dropna(inplace=True)

print(df.shape)

In [ ]:
# =========================
# TEXT PREPROCESSING
# =========================

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenization
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return ' '.join(words)

df[text_column] = df[text_column].apply(preprocess_text)

print(df.head())

In [ ]:
# =========================
# ENCODE TARGET LABELS
# =========================

# Example:
# positive -> 1
# negative -> 0

df[target_column] = df[target_column].map({
    'positive': 1,
    'negative': 0
})

print(df[target_column].value_counts())

In [ ]:
# =========================
# TRAIN TEST SPLIT
# =========================

X = df[text_column]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

## Machine Learning Technique - Logistic Regression

In [ ]:
# =========================
# TF-IDF VECTORIZATION
# =========================

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)

In [ ]:
# =========================
# LOGISTIC REGRESSION MODEL
# =========================

lr_model = LogisticRegression()

# Train model
lr_model.fit(X_train_tfidf, y_train)

# Predictions
y_pred = lr_model.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

## Deep Learning Technique - LSTM

In [ ]:
# =========================
# TOKENIZATION
# =========================

max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len
)

print(X_train_pad.shape)

In [ ]:
# =========================
# BUILD LSTM MODEL
# =========================

lstm_model = Sequential()

lstm_model.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    )
)

lstm_model.add(LSTM(64))

lstm_model.add(Dense(1, activation='sigmoid'))

# Compile model
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

In [ ]:
# =========================
# TRAIN LSTM MODEL
# =========================

history = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
# =========================
# EVALUATE LSTM MODEL
# =========================

loss, accuracy = lstm_model.evaluate(
    X_test_pad,
    y_test
)

print("\nTest Accuracy:", accuracy)

# Predictions
predictions = lstm_model.predict(X_test_pad)

predicted_classes = (predictions > 0.5).astype(int)

print("\nPredicted Classes:")
print(predicted_classes[:10])

## Conclusion

- Logistic Regression performs well for text classification using TF-IDF.
- LSTM captures sequential text patterns and deep semantic relationships.
- This project demonstrates both ML and DL approaches for sentiment analysis.